# Lab 1: Learning simple probability distributions

A **generative model** is a probability distribution $p(\mathbf{x})$ over data that we *learn from examples* and can then *sample from*. In this lab you build the simplest such models yourself and see where they work and where they break.

For every model you go through the same three steps:

| Step | Question |
|---|---|
| **Learn** | Which parameters of the model make it describe the training examples? |
| **Generate** | Can we draw new examples from the learned model, and do they look like the data? |
| **Evaluate** | How well does the model describe *new* data it has not seen during learning? |

## Overview

| Part | Data | Models |
|---|---|---|
| **A** | 1 variable: waiting times | histogram, single Gaussian, mixture of Gaussians (EM) |
| **B** | 2 variables: waiting time and bill | 2D histogram, independent vs. joint Gaussian, 2D mixture |
| **C** | 2D moons | mixtures with a growing number of components |
| **D** (optional) | images: MNIST digits | independent pixels |

## How to work

- **The code you write lives in scripts**, not in this notebook: `histogram.py`, `gaussian.py`, `mixture.py` and `pixels.py` in the folder `week01/`. Open them in an editor (e.g. VS Code). Every function has a docstring that says what goes in and what comes out; the bodies are marked `# TODO`.
- **This notebook only presents**: it imports your functions, runs checks, draws plots and evaluates the models. Do not put your solutions here.
- After saving a script, simply rerun the notebook cell - the next cell reloads your scripts automatically.
- `check(...)` lines tell you whether a function passes a basic test. Passing the checks does not prove the code is correct - look at the plots.
- Each evaluation uses only the output of its own task, so an unfinished task does not break the others. Only B3 and C also need your EM code from A3.
- Write your answers to the questions in the markdown cells marked **Answer**.
- Everything is done with `torch` tensors in 64-bit precision, with the random seed fixed so that results are reproducible.

## How we evaluate

A model is fitted on the *training* examples and judged on *test* examples that were not used for fitting. The score is the **negative log-likelihood (NLL)**:

$$\mathrm{NLL} = -\frac{1}{M}\sum_{i=1}^{M} \log p(\mathbf{x}_i)$$

averaged over $M$ test examples, measured in *nats* (natural logarithm).

- **Lower is better**: a good model gives the test examples a high density.
- For continuous variables $p(\mathbf{x})$ is a density and can exceed 1, so the NLL can be negative. Compare NLLs of different models on the *same* data only.
- If a model gives a test example probability zero, then the NLL is infinite.

In [ ]:
# Make the course repository importable (no installation needed): its root holds the helper folder `udl/`
# and the lab folders such as `week01/`. Locally the root is found above this notebook. On Colab the repository is cloned.
import subprocess
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "udl").is_dir()), None)
if root is None:  # not inside the repository, e.g. Colab
    REPO = "https://gitlab.example.com/CHANGE-ME/udl-student.git"  # TODO(lecturer): student repo URL
    root = Path("udl-student").resolve()
    if not root.exists():
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=True)
sys.path.insert(0, str(root))

In [ ]:
# Reload the scripts automatically after you edit and save them
%load_ext autoreload
%autoreload 2

In [ ]:
import math

import torch
import matplotlib.pyplot as plt

# provided helpers
from udl.datasets import make_dataset, load_mnist
from udl.distributions import log_gaussian, gmm_logpdf
from udl.lab01 import make_waiting_times, make_orders_2d, hist_logpdf, hist2d_logpdf, bernoulli_logpdf, plot_1d_fit, plot_2d_fit
from udl.plotting import plot_images
from udl.utils import set_seed, check, report

# your code (scripts in the folder `week01/`)
from week01.histogram import fit_histogram, sample_histogram, fit_histogram_2d, sample_histogram_2d
from week01.gaussian import fit_gaussian, sample_gaussian, remove_correlations
from week01.mixture import e_step, m_step, gmm_em, sample_gmm, n_params
from week01.pixels import fit_pixel_probs, sample_pixels

torch.set_default_dtype(torch.float64)
set_seed(0)

# A. One variable: waiting times

We observe how long (in minutes) guests of a restaurant wait for their dish. There seem to be two kinds of guests - those with quick dishes and those with slow ones. We have **200 training examples** to learn from and **1000 test examples** for evaluation.

In [ ]:
x_all = make_waiting_times(1200, seed=0)
x_train, x_test = x_all[:200], x_all[200:]

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(x_train.numpy(), bins=28, range=(0, 14), density=True, color="#a6bddb", edgecolor="#163c69")
ax.set_xlabel("waiting time [min]")
ax.set_title("training data")
plt.show()

## A1. Histogram

**Model.** Split the range into $B$ bins with edges $e_0 < e_1 < \dots < e_B$. Bin $b$ gets probability $\theta_b$, with $\sum_b \theta_b = 1$. Inside a bin the density is constant, so for $e_b \le x < e_{b+1}$

$$p(x) = \frac{\theta_b}{e_{b+1} - e_b}$$

**Learn.** The probability of a bin is the share of training examples that fall into it.

**Generate.** Choose a bin with probability $\theta_b$, then a point uniformly inside it.

> **Your task** (`histogram.py`): implement `fit_histogram` and `sample_histogram`.

In [ ]:
edges = torch.linspace(0, 14, 15)  # 14 bins of width 1 - you will change this below

probs = fit_histogram(x_train, edges)
check("bin probabilities sum to 1", abs(float(probs.sum()) - 1) < 1e-6)
samples = sample_histogram(probs, edges, 5000)
check("samples stay inside the range", samples.min() >= edges[0] and samples.max() <= edges[-1])
check("samples have about the mean of the data", abs(float(samples.mean() - x_train.mean())) < 0.3)

fig, ax = plt.subplots(figsize=(5, 3))
plot_1d_fit(ax, x_train, samples, title="histogram")
plt.show()

**Evaluate.** The density $p(x)$ of the histogram is provided in `hist_logpdf`.

In [ ]:
report("A1 histogram", hist_logpdf(x_test, probs, edges))

**Answer A1.** Change the number of bins (for example `torch.linspace(0, 14, 8)` and `torch.linspace(0, 14, 57)`) and rerun A1. What happens to the plot and to the NLL? Why can the NLL be infinite?

*(your answer here)*

## A2. Single Gaussian

**Model.** A bell shape with two parameters, the mean $\mu$ and the variance $\sigma^2$. We write it for $d$ variables right away - mean vector $\boldsymbol\mu$ and covariance matrix $\boldsymbol\Sigma$:

$$p(\mathbf{x}) = \mathcal{N}(\mathbf{x} \mid \boldsymbol\mu, \boldsymbol\Sigma)$$

**Learn.** The sample mean and the sample covariance of the training examples,

$$\hat{\boldsymbol\mu} = \frac{1}{N}\sum_i \mathbf{x}_i, \qquad \hat{\boldsymbol\Sigma} = \frac{1}{N}\sum_i (\mathbf{x}_i - \hat{\boldsymbol\mu})(\mathbf{x}_i - \hat{\boldsymbol\mu})^\top$$

**Generate.** Let $\boldsymbol\Sigma = \mathbf{L}\mathbf{L}^\top$ (a *Cholesky factorisation*, $\mathbf{L}$ lower triangular). If $\mathbf{z}$ is a vector of independent standard normal numbers, then $\boldsymbol\mu + \mathbf{L}\mathbf{z}$ has distribution $\mathcal{N}(\boldsymbol\mu, \boldsymbol\Sigma)$.

> **Your task** (`gaussian.py`): implement `fit_gaussian` and `sample_gaussian`. They work for any number of variables $d$; a single variable is the case $d = 1$, data of shape `(N, 1)`.

In [ ]:
X_train = x_train[:, None]  # (N, 1): one variable
mean, cov = fit_gaussian(X_train)
check("mean is correct", abs(float(mean[0] - x_train.mean())) < 1e-6)
check("variance is correct (dividing by N)", abs(float(cov[0, 0] - x_train.var(unbiased=False))) < 1e-6)
samples = sample_gaussian(mean, cov, 5000)
check("samples have the fitted mean and spread", abs(float(samples.mean() - mean[0])) < 0.2 and abs(float(samples.std() - cov[0, 0].sqrt())) < 0.2)

logpdf_gauss = lambda x: log_gaussian(x[:, None], mean, cov)
fig, ax = plt.subplots(figsize=(5, 3))
plot_1d_fit(ax, x_train, samples[:, 0], logpdf_gauss, title="single Gaussian")
plt.show()

**Evaluate.** `log_gaussian(X, mean, cov)` is provided; it works for any number of variables.

In [ ]:
report("A2 single Gaussian", logpdf_gauss(x_test))

**Answer A2.** Look at the samples and at the model density. Where does the single Gaussian put probability that the data does not support?

*(your answer here)*

## A3. Mixture of Gaussians fitted by EM

**Model.** A weighted sum of $K$ Gaussians, the *components*:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol\mu_k, \boldsymbol\Sigma_k), \qquad \pi_k \ge 0, \quad \sum_k \pi_k = 1$$

**Generate.** First choose a component $k$ with probability $\pi_k$, then sample $\mathbf{x}$ from $\mathcal{N}(\boldsymbol\mu_k, \boldsymbol\Sigma_k)$.

**Learn.** If we knew which component generated each example, we would take the share of examples, their mean and their covariance in every group. We do not know it. The **EM algorithm** therefore works with *soft* assignments, the **responsibilities**

$$r_{ik} = \text{probability that example } i \text{ was generated by component } k$$

and alternates between two steps, starting from some initial parameters:

| Step | Computes | Formula |
|---|---|---|
| **E-step** | responsibilities for the current parameters | $\displaystyle r_{ik} = \frac{\pi_k\,\mathcal{N}(\mathbf{x}_i \mid \boldsymbol\mu_k, \boldsymbol\Sigma_k)}{\sum_{j} \pi_j\,\mathcal{N}(\mathbf{x}_i \mid \boldsymbol\mu_j, \boldsymbol\Sigma_j)}$ |
| **M-step** | new parameters for the current responsibilities, with $N_k = \sum_i r_{ik}$ | $\displaystyle \pi_k = \frac{N_k}{N}, \quad \boldsymbol\mu_k = \frac{1}{N_k}\sum_i r_{ik}\,\mathbf{x}_i, \quad \boldsymbol\Sigma_k = \frac{1}{N_k}\sum_i r_{ik}\,(\mathbf{x}_i - \boldsymbol\mu_k)(\mathbf{x}_i - \boldsymbol\mu_k)^\top$ |

The log-likelihood $\sum_i \log p(\mathbf{x}_i)$ of the training data never decreases during EM - a useful test of your code.

**Hints.** Work with logarithms ($\log \pi_k + \log\mathcal{N}(\dots)$) and normalise with a log-sum-exp, otherwise the numbers underflow. The provided `log_gaussian` gives the log-density of one Gaussian. Reference: Bishop, *Pattern Recognition and Machine Learning*, Section 9.2.2.

> **Your task** (`mixture.py`): implement `e_step`, `m_step`, the loop in `gmm_em` and `sample_gmm`. Write them for any number of variables $d$ - you will reuse them in B and C. The initial parameters come from the provided `init_gmm`.

In [ ]:
pis, mus, Sigmas, history = gmm_em(X_train, k=2, n_iter=50)
check("log-likelihood never decreases", all(b >= a - 1e-8 for a, b in zip(history, history[1:])), "a correct EM only goes up - test the E- and the M-step separately")
check("weights sum to 1", abs(float(pis.sum()) - 1) < 1e-6)
check("component means are close to 3.3 and 8.0", torch.allclose(torch.sort(mus.flatten()).values, torch.tensor([3.3, 8.0]), atol=0.5))
samples = sample_gmm(pis, mus, Sigmas, 5000)
check("samples have shape (n, 1)", tuple(samples.shape) == (5000, 1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
plot_1d_fit(axes[0], x_train, samples[:, 0], lambda x: gmm_logpdf(x[:, None], pis, mus, Sigmas), title="mixture of 2 Gaussians")
axes[1].plot(history)
axes[1].set_xlabel("EM iteration")
axes[1].set_ylabel("log-likelihood of training data")
plt.tight_layout()
plt.show()

**Evaluate.** `gmm_logpdf` is provided.

In [ ]:
report("A3 mixture of 2 Gaussians", gmm_logpdf(x_test[:, None], pis, mus, Sigmas))

**Answer A3.** Compare the three NLLs of A1, A2 and A3. Which model is best? Rerun A3 with `seed=1`, `2`, `3` in `gmm_em`: do you always get the same result? Why?

*(your answer here)*

# B. Two variables: waiting time and bill

Now every guest has two numbers: the waiting time and the bill. Within each of the two groups of guests a longer wait goes with a bigger bill. Again 200 training and 1000 test examples.

In [ ]:
X2_all = make_orders_2d(1200, seed=1)
X2_train, X2_test = X2_all[:200], X2_all[200:]

fig, ax = plt.subplots(figsize=(4, 3.5))
ax.scatter(X2_train[:, 0].numpy(), X2_train[:, 1].numpy(), s=8, color="#729fcf")
ax.set_xlabel("waiting time [min]")
ax.set_ylabel("bill [EUR]")
plt.show()

## B1. 2D histogram

**Model.** The same idea as A1 in two dimensions: split both axes into bins, which gives a grid of *cells*. Every cell gets a probability, all probabilities sum to 1, and the density inside a cell is constant: cell probability divided by cell area. We use 10 bins per variable, i.e. 100 cells.

**Learn.** The probability of a cell is the share of training examples in it. **Generate.** Choose a cell, then a point uniformly inside it.

> **Your task** (`histogram.py`): implement `fit_histogram_2d` and `sample_histogram_2d`.

In [ ]:
edges_x = torch.linspace(0, 12, 11)
edges_y = torch.linspace(0, 60, 11)

probs2 = fit_histogram_2d(X2_train, edges_x, edges_y)
check("cell probabilities sum to 1", abs(float(probs2.sum()) - 1) < 1e-6)
check("grid of 10 x 10 cells", tuple(probs2.shape) == (10, 10))
samples = sample_histogram_2d(probs2, edges_x, edges_y, 500)
check("samples have shape (n, 2)", tuple(samples.shape) == (500, 2))
print(f"{int((probs2 > 0).sum())} of {probs2.numel()} cells contain at least one training example")

fig, ax = plt.subplots(figsize=(4.5, 4))
plot_2d_fit(ax, X2_train, samples, title="2D histogram")
plt.show()

**Evaluate.** `hist2d_logpdf` is provided.

In [ ]:
report("B1 2D histogram", hist2d_logpdf(X2_test, probs2, edges_x, edges_y))

**Answer B1.** How many test examples have probability zero? Compare with the 1D histogram in A1. What would happen with three variables and the same number of training examples?

*(your answer here)*

## B2. Independent versus joint Gaussian

Two single-Gaussian models for the two variables:

- **Joint Gaussian:** one Gaussian over both variables with a full covariance matrix $\boldsymbol\Sigma$. Its off-diagonal entries describe how the variables vary together.
- **Independent Gaussians:** every variable gets its own Gaussian, unrelated to the other. This is the same as a joint Gaussian whose covariance matrix has zeros off the diagonal.

> **Your task:** `fit_gaussian` and `sample_gaussian` from A2 already work for two variables. In `gaussian.py` implement `remove_correlations`.

In [ ]:
mean2, cov_full = fit_gaussian(X2_train)
cov_indep = remove_correlations(cov_full)
check("independent model keeps the variances", torch.allclose(torch.diagonal(cov_indep), torch.diagonal(cov_full)))
check("independent model has zero covariance", abs(float(cov_indep[0, 1])) < 1e-12 and abs(float(cov_indep[1, 0])) < 1e-12)

samples_full = sample_gaussian(mean2, cov_full, 500)
samples_indep = sample_gaussian(mean2, cov_indep, 500)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_2d_fit(axes[0], X2_train, samples_full, title="joint Gaussian")
plot_2d_fit(axes[1], X2_train, samples_indep, title="independent Gaussians")
plt.show()

**Evaluate.**

In [ ]:
report("B2 joint Gaussian", log_gaussian(X2_test, mean2, cov_full))
report("B2 independent Gaussians", log_gaussian(X2_test, mean2, cov_indep))

**Answer B2.** What does the independent model lose - in the plot and in the NLL? How many free parameters does each of the two models have?

*(your answer here)*

## B3. 2D mixture of Gaussians

The functions `gmm_em` and `sample_gmm` from A3 work for any number of variables. Fit a mixture of 2 Gaussians to the two-variable data.

In [ ]:
pis2, mus2, Sigmas2, history2 = gmm_em(X2_train, k=2, n_iter=100)
check("log-likelihood never decreases", all(b >= a - 1e-8 for a, b in zip(history2, history2[1:])))
samples = sample_gmm(pis2, mus2, Sigmas2, 500)

fig, ax = plt.subplots(figsize=(4.5, 4))
plot_2d_fit(ax, X2_train, samples, title="mixture of 2 Gaussians")
plt.show()

**Evaluate.**

In [ ]:
report("B3 mixture of 2 Gaussians", gmm_logpdf(X2_test, pis2, mus2, Sigmas2))

**Answer B3.** Compare the NLLs of the four 2D models (B1, B2 joint, B2 independent, B3). Which one would you choose, and why?

*(your answer here)*

# C. Moons: how many Gaussians do we need?

Two interleaved half circles - nothing like a bell shape. We fit mixtures with a growing number of components $K$ and look at the samples, at the test NLL and at the number of parameters.

A mixture of $K$ Gaussians with full covariance matrices in $d$ dimensions has $K$ means ($d$ numbers each), $K$ covariance matrices ($d(d+1)/2$ free numbers each, because the matrix is symmetric) and $K$ weights that sum to 1 ($K - 1$ free numbers).

> **Your task** (`mixture.py`): implement `n_params`.

In [ ]:
moons = make_dataset("moons", n=3000, seed=0).double()
M_train, M_test = moons[:2000], moons[2000:]

check("2 components in 2D: 11 parameters", n_params(2, 2) == 11)

ks = [2, 5, 10, 20]
fig, axes = plt.subplots(1, len(ks), figsize=(3.4 * len(ks), 3.4))
for ax, k in zip(axes, ks):
    pis_k, mus_k, Sigmas_k, _ = gmm_em(M_train, k, n_iter=200)
    samples = sample_gmm(pis_k, mus_k, Sigmas_k, 1000)
    plot_2d_fit(ax, M_train[:500], samples, title=f"K = {k}  ({n_params(k, 2)} parameters)", xlim=(-3, 3), ylim=(-3, 3), labels=("x1", "x2"))
    report(f"C moons, K = {k:2d}", gmm_logpdf(M_test, pis_k, mus_k, Sigmas_k))
plt.show()

**Answer C.** From which $K$ on do the samples look like the moons? What does that cost in parameters and what do you gain in NLL? What property of the Gaussian components makes it so expensive to describe such a simple shape, and what would you want instead?

*(your answer here)*

# D. Images: independent pixels on MNIST (optional)

**Data.** MNIST: $28 \times 28$ images of handwritten digits. We threshold every pixel to *on* (1) or *off* (0), so an image is a vector of $D = 784$ binary variables. The first call downloads the data.

**Model: independent pixels.** Every pixel $j$ has its own coin with probability $p_j$ of being on, unrelated to all other pixels:

$$p(\mathbf{x}) = \prod_{j=1}^{784} p_j^{x_j}\,(1 - p_j)^{1 - x_j}$$

**Learn.** $p_j$ is the share of training images in which pixel $j$ is on. **Generate.** Flip every pixel's coin separately.

> **Your task** (`pixels.py`): implement `fit_pixel_probs` and `sample_pixels`.

In [ ]:
digits_train, digits_test = load_mnist()
B_train = (digits_train.reshape(len(digits_train), -1) > 0.5).double()  # (60000, 784)
B_test = (digits_test.reshape(len(digits_test), -1) > 0.5).double()

p = fit_pixel_probs(B_train)
check("one probability per pixel", tuple(p.shape) == (784,))
samples = sample_pixels(p, 16)
check("samples have shape (n, 784) and are binary", tuple(samples.shape) == (16, 784) and bool(((samples == 0) | (samples == 1)).all()))

fig, axes = plt.subplots(2, 8, figsize=(8, 2.2))
plot_images(axes[0], B_train[:8], title="real")
plot_images(axes[1], samples[:8], title="independent pixels")
plt.show()

**Evaluate.** `bernoulli_logpdf` is provided. For images the NLL is usually quoted in **bits per pixel**: divide the NLL in nats by $784 \cdot \ln 2$. A model that knows nothing - all $2^{784}$ binary images equally likely - needs exactly **1 bit per pixel**.

In [ ]:
nll = report("D independent pixels", bernoulli_logpdf(B_test, p))
print(f"bits per pixel: {nll / (784 * math.log(2)):.3f}   (all images equally likely: 1.000)")

**Answer D.** Why do the samples look like noise although the model knows which pixels are usually on? What kind of structure is missing? How would a histogram model work on this data?

*(your answer here)*